In [ ]:
# --- cell 1 · bootstrap（任务书 §0.1 末、§0.2）—— 五个 B notebook 里这一格完全相同 ---
import pathlib, sys

# (1) 向上搜索含 npnn/ 的目录 —— 不依赖 Jupyter 的启动目录
def _find_taskb_root(start):
    for p in (start, *start.parents):
        if (p / 'npnn').is_dir():
            return p
        if (p / 'task_b' / 'npnn').is_dir():      # 从项目根启动 Jupyter 的情况
            return p / 'task_b'
    return None

_here = pathlib.Path.cwd().resolve()
TASKB_ROOT = _find_taskb_root(_here)
assert TASKB_ROOT is not None, f'从 {_here} 逐级往上找不到含 npnn/ 的目录；请从 task_b/ 或项目根启动'
sys.path.insert(0, str(TASKB_ROOT))

# (2) 物理挡住「误 import A 的 src」：把项目根从 sys.path 里摘掉（任务书 §0.1.2）
_root = TASKB_ROOT.parent
sys.path[:] = [e for e in sys.path if pathlib.Path(e or '.').resolve() != _root]

# (3) 线程预算：必须在 import numpy 之前（BLAS 首次 import 时锁死线程数）
import config
config.limit_threads()

# (4) 环境留痕 —— 报告 B.1 要用：哪个解释器、哪个 numpy
import numpy as np
print('sys.executable:', sys.executable)
print('sys.prefix    :', sys.prefix)
print('numpy         :', np.__version__)
print('TASKB_ROOT    :', TASKB_ROOT)


# 11 · 用自己的 npnn 端到端训练（第 3 阶段）

对应任务书：§3.1（三级金字塔的 L3）、§3.3（性能工程）、§3.4（网络设计）、§0.5（计时预算）。

## 网络（3 个 stage，早期下采样 —— 借 Task A 的架构直觉）
```
3x32x32 --conv3x3--> 32x30x30 --SN,GELU-->
32x30x30 --conv3x3 s2--> 64x14x14 --SN,GELU-->
64x14x14 --conv3x3 s2--> 128x6x6 --SN,GELU-->
GAP --> 128 --Linear--> 100
```
params 107,268；全程我们自己的 Conv2d / SwitchNorm2d / GELU / Linear / CE / Adam。

## 本 notebook 的产出
- [x] **cell 2**：跑 `tests/train_npnn.py`（**已有结果就跳过，不重训**）。全量 45k train / 5k val，
      20 epoch，batch 128，Adam lr=1e-3，float32，4 线程。产物落 `runs/cnn_sn/`：
      `config.json / metrics.csv / sn_weights.json / params.npz`
- [x] **cell 3**：报告图 —— 训练曲线（`report/figs/train_curves.png`）+
      **SN 权重演化**（`report/figs/sn_weight_evolution.png`）

## 性能记录（实测，写进报告）
- 计时基线（§0.5）低估了 conv 的成本：它只量了"拷贝 + GEMM"的**下界**，没算 im2col 的跨步
  gather 和输出转置。真实单层前向 ≈ 4× 基线值。
- 训练时最大的热点是 **GELU 的 `x ** 3`**：numpy 对整数幂 3 走通用 libm pow，单层前向 57 ms。
  改成 `x*x*x` 后降到 30 ms，整轮 epoch 从 320 s → 257 s。
- 线程数必须用 `config.limit_threads()` 在 **import numpy 之前**设置（训练脚本已修）。

⏳ 训练很慢（≈4.3 min/epoch）。**越早放出去跑越好**。
降级预案（§时间安排末）：子集训练 / 减 epoch —— 但 L2 整网对拍必须保留（已在 notebook 12 完成）。

## 训练前先过这三关

1. **数值梯度 + torch 对拍都通过了**（第 2 阶段验收），再谈训练 —— 否则是在错误的地基上盖楼
2. **单一变量**：先跑通「无 SN 的小网」建立基线，再加 SN / 加宽加深，别一次改三样
3. **同预算锚点**：和要对比的 torch 侧模型用同一份 config（架构、epoch、lr、batch）

⚠️ 与 A 抢 CPU：`OMP_NUM_THREADS` 限到 4–8（先量 1/2/4/8/16 的拐点）。A 的 dataloader 要用 CPU。
⚠️ dtype：训练用 float32；只有对拍/梯度检验用 float64 —— 模块内部别硬编码 dtype，跟着输入走。


In [ ]:
# --- cell 2 · 端到端训练（脚本 tests/train_npnn.py）---
# ⏳ 约 4.3 分钟/epoch。已经跑过就直接用落盘结果，**不重训**（Run All 才不会变成一小时）。
#    重新训练：在终端跑  python tests/train_npnn.py --epochs 20 --out cnn_sn
import subprocess, sys
from config import RUNS_DIR

run_dir = RUNS_DIR / 'cnn_sn'
if (run_dir / 'metrics.csv').exists():
    print('已有训练结果，跳过重训 ->', run_dir)
    print('（要新结果就先删掉这个目录，或换 --out 名字）')
else:
    r = subprocess.run([sys.executable, str(TASKB_ROOT / 'tests' / 'train_npnn.py'),
                        '--epochs', '20', '--out', 'cnn_sn'],
                       capture_output=True, text=True, encoding='utf-8', errors='replace')
    print(r.stdout)
    print('exit code:', r.returncode)

In [ ]:
# --- cell 3 · 报告图：训练曲线 + SwitchNorm 权重演化 ---
# 两张图都只读 runs/cnn_sn/ 里落盘的数据，不重训
import csv
import json
import matplotlib.pyplot as plt
from config import FIG_DIR, RUNS_DIR

run_dir = RUNS_DIR / 'cnn_sn'
with open(run_dir / 'metrics.csv', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
ep = [int(r['epoch']) for r in rows]
tr_l, va_l = [float(r['train_loss']) for r in rows], [float(r['val_loss']) for r in rows]
tr_a, va_a = [float(r['train_acc']) for r in rows], [float(r['val_acc']) for r in rows]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.9))
ax1.plot(ep, tr_l, label='train'); ax1.plot(ep, va_l, label='val')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, tr_a, label='train'); ax2.plot(ep, va_a, label='val')
ax2.set_xlabel('epoch'); ax2.set_ylabel('accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'train_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# --- SN 权重演化：论文核心主张的直接证据 ---
sn = json.loads((run_dir / 'sn_weights.json').read_text(encoding='utf-8'))
epochs, hist = sn['epochs'], sn['hist']          # hist[e][layer] = [w_bn, w_in, w_ln]
n_layers = len(hist[0])
fig2, axes = plt.subplots(1, n_layers, figsize=(4.2 * n_layers, 3.6), squeeze=False)
for k in range(n_layers):
    ax = axes[0, k]
    for j, name in enumerate(('BN', 'IN', 'LN')):
        ax.plot(epochs, [h[k][j] for h in hist], marker='o', ms=3, label=name)
    ax.set_title(f'SN layer {k + 1}'); ax.set_xlabel('epoch'); ax.set_ylim(0, 1)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0, 0].set_ylabel(r'$w = \mathrm{softmax}(\lambda)$')
fig2.tight_layout()
fig2.savefig(FIG_DIR / 'sn_weight_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'final: train {tr_a[-1]:.1%}  val {va_a[-1]:.1%}')
print('last-epoch SN weights (BN, IN, LN):')
for k, w in enumerate(hist[-1]):
    print(f'  layer {k + 1}: {w[0]:.3f}  {w[1]:.3f}  {w[2]:.3f}')